<a href="https://colab.research.google.com/github/melissa-04/melisayla-biyoinformatik/blob/main/notebooks/rna-seq/05_kara_kutu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kara kutuyu aç: pseudoalignment ne yapar?

Bu defterde veri indirmiyoruz; Salmon'un içindeki fikri, elle sayabileceğiniz kadar küçük bir oyuncakla kuruyoruz. Üç transkript, dört okuma, k=5. Gerçekte sayılar milyonlarca ve k=31; mantık birebir aynı.

In [1]:
k = 5

transkriptler = {
    'T1': 'ATGGCTACGATCGGCTAAGCTTGA',
    'T2': 'ATGGCTACGATCGGCTCCGATTGA',   # başı T1 ile ortak (paylaşılan ekzon)
    'T3': 'GGTACCTTGACGTAGCATCGATCA',   # bambaşka
}

# 1) İndeks: her k-mer hangi transkriptlerde geçiyor?
indeks = {}
for ad, dizi in transkriptler.items():
    for i in range(len(dizi) - k + 1):
        indeks.setdefault(dizi[i:i+k], set()).add(ad)
print('İndeksteki k-mer sayısı:', len(indeks))

İndeksteki k-mer sayısı: 45


## Eşleştirme

Bir okumayı harf harf hizalamak yerine sorumuz şu: okumanın bütün k-mer'leri hangi transkriptlerde *birden* geçiyor? Kesişim boşsa okuma eşleşmedi demektir; adaptör kuyruklarının eşleşememesi tam bu yüzden.

In [2]:
# Okumalar: dizileyiciden gelmiş gibi
okumalar = {
    'okuma1': 'CGGCTAAGC',   # T1'e özgü bölge
    'okuma2': 'ATGGCTACG',   # T1 ile T2'nin ortak başı
    'okuma3': 'GACGTAGCA',   # T3'e özgü
    'okuma4': 'GCTACGATC',   # yine ortak bölge
}

# 3) Eşleştirme: okumanın k-mer'leri hangi transkriptlerde ortak?
uyum = {}
for ad, dizi in okumalar.items():
    kumeler = [indeks.get(dizi[i:i+k], set()) for i in range(len(dizi) - k + 1)]
    uyum[ad] = set.intersection(*kumeler) if kumeler else set()
    print(ad, '->', sorted(uyum[ad]))

okuma1 -> ['T1']
okuma2 -> ['T1', 'T2']
okuma3 -> ['T3']
okuma4 -> ['T1', 'T2']


## Paylaştırma: tanıklar oy kullanıyor

okuma2 ve okuma4 kararsız: T1 de olabilir, T2 de. Ama okuma1 yalnız T1'e uyuyor; o bir tanık. EM döngüsü, tanıkların oyunu kararsız okumalara yansıtır: T1'in bolluk tahmini arttıkça kararsız okumaların payı da T1'e kayar. Turları izleyin.

In [3]:
# EM: belirsiz okumaları olasılıkla paylaştır
bolluk = {t: 1/3 for t in transkriptler}
for tur in range(1, 6):
    pay = {t: 0.0 for t in transkriptler}
    for ad, adaylar in uyum.items():
        toplam = sum(bolluk[t] for t in adaylar)
        for t in adaylar:
            pay[t] += bolluk[t] / toplam
    bolluk = {t: pay[t] / len(okumalar) for t in transkriptler}
    print(f'tur {tur}:', {t: round(pay[t], 2) for t in pay})

tur 1: {'T1': 2.0, 'T2': 1.0, 'T3': 1.0}
tur 2: {'T1': 2.33, 'T2': 0.67, 'T3': 1.0}
tur 3: {'T1': 2.56, 'T2': 0.44, 'T3': 1.0}
tur 4: {'T1': 2.7, 'T2': 0.3, 'T3': 1.0}
tur 5: {'T1': 2.8, 'T2': 0.2, 'T3': 1.0}


## Kendin dene

Üç deney: (1) okumalardan birini bozun (bir harf değiştirin); eşleşme ne oluyor, neden bu kadar kırılgan? Gerçek Salmon'un tek harf hatasına bizim oyuncaktan daha dayanıklı olduğunu aklınızda tutun. (2) T3'e özgü ikinci bir okuma ekleyin; bolluklar nasıl değişiyor? (3) okuma1'i silin: tanık yokken EM kararsız okumaları nasıl paylaştırıyor? Bu üçüncüsü, "izoform kantifikasyonu neden zordur" sorusunun cevabıdır.